In [1]:
import torch
import torch.nn as nn 

In [27]:


inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

In [28]:
query =inputs[1]
simpl_attent = torch.empty(inputs.shape[0])
for i,xi in enumerate(inputs):
    simpl_attent[i] = torch.dot(xi,query)
    
print(simpl_attent)



tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [29]:
atten_soft = torch.softmax(simpl_attent, dim=0)

context_result  = torch.zeros(query.shape)

for i, xi in enumerate(inputs):
    print(f'{i} -> {xi}')
    print(f'attention ->  {atten_soft[i]*xi}')
    context_result += atten_soft[i]*xi
    
print(context_result)


0 -> tensor([0.4300, 0.1500, 0.8900])
attention ->  tensor([0.0596, 0.0208, 0.1233])
1 -> tensor([0.5500, 0.8700, 0.6600])
attention ->  tensor([0.1308, 0.2070, 0.1570])
2 -> tensor([0.5700, 0.8500, 0.6400])
attention ->  tensor([0.1330, 0.1983, 0.1493])
3 -> tensor([0.2200, 0.5800, 0.3300])
attention ->  tensor([0.0273, 0.0719, 0.0409])
4 -> tensor([0.7700, 0.2500, 0.1000])
attention ->  tensor([0.0833, 0.0270, 0.0108])
5 -> tensor([0.0500, 0.8000, 0.5500])
attention ->  tensor([0.0079, 0.1265, 0.0870])
tensor([0.4419, 0.6515, 0.5683])


## attention mechanism 

In [72]:
attetion_score = inputs @ inputs.T
attetion_score

atten_soft = torch.softmax(attetion_score , dim =-1)
atten_soft




tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [31]:
print(atten_soft.shape , inputs.shape)
context_result = atten_soft @ inputs
context_result

torch.Size([6, 6]) torch.Size([6, 3])


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [117]:
torch.manual_seed(1337)
class SelfAttention(nn.Module) :
    def __init__(self, d_in ,d_out ,bias =False):
        super().__init__()
        self.k_weights = nn.Linear(d_in,d_out , bias=bias)
        self.q_weights = nn.Linear(d_in,d_out ,bias=bias)
        self.v_weights = nn.Linear(d_in,d_out,bias=bias)
        
    def forward(self ,x):
        keys =  self.k_weights(x)
        query =  self.q_weights(x)
        values = self.v_weights(x)
        keys =torch.transpose(keys ,-2,-1 )
        # print(keys.shape)
        
        
        # print(keys.transpose(keys.shape[-2] , keys.shape[-1]))
        # print(query.shape)
        attention = query @ keys
        # print(f'attentio shape -->{attention.shape}')
        # atten_soft = torch.softmax(attention/keys.shape[-1] ** 0.5,dim=-1)
        context_length = attention.shape[-1]
        mask = torch.tril(torch.ones(context_length,context_length), diagonal=0)
        # print(f'mask shape -->{mask.shape}')
        
        # print(mask)
        masked_attention = attention.masked_fill(mask.bool() == False , -torch.inf)
        masked_attention = torch.softmax(masked_attention/keys.shape[-1],dim=-1)
        # print(f'mask attentio shape -->{masked_attention.shape}')
        
        # print(masked_attention)
        #  adding dropout layer 
        dropout = nn.Dropout(0.5)
        dropout_attention= dropout(masked_attention)
        # print(dropout_attention)
        # print(f'drop out attentio shape -->{dropout_attention.shape}')
        
        
        context_vec = dropout_attention @ values
        
        return context_vec
d_in = inputs.shape[1]
d_out =2
sa_v1 = SelfAttention(d_in,d_out)
# context_result = sa_v1(inputs)
# context_result
        

In [118]:
batch = torch.stack((inputs,inputs))
# batch.shape
context_result2 = sa_v1(batch)
context_result2.shape

torch.Size([2, 6, 2])

In [123]:


inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your
[0.55, 0.87, 0.66], # journey
[0.57, 0.85, 0.64], # starts
[0.22, 0.58, 0.33], # with
[0.77, 0.25, 0.10], # one
[0.05, 0.80, 0.55]] # step
)

class Casual_Attention(nn.Module):
    def __init__(self,d_in ,d_out ,context_length,dropout, bias =False  ):
        super().__init__()
        self.d_out = d_out,
        self.d_in = d_in
        self.k_weights = nn.Linear(d_in,d_out , bias )
        self.q_weights = nn.Linear(d_in,d_out , bias )
        self.v_weights = nn.Linear(d_in,d_out , bias )
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask' ,torch.tril(torch.ones(context_length,context_length),diagonal=0))
    
    def forward(self, x):
        keys = self.k_weights(x)
        query = self.q_weights(x)
        values = self.v_weights(x)
        # keys = torch.transpose(keys , -1 ,-2)
        
        attention_score = query @ keys.transpose(1,2)
        
        masked_attention = attention_score.masked_fill(self.mask ==0 , -torch.inf)
        norm_masked_attention = torch.softmax(masked_attention/keys.shape[-1]**0.5 , dim=-1 )
        # print(norm_masked_attention)
        norm_masked_attention = self.dropout(norm_masked_attention)
        
        context_result = norm_masked_attention  @ values
        return context_result
        
    

In [124]:
ca =Casual_Attention(3,2, len(inputs) ,0.2)

batch = torch.stack((inputs,inputs))
# batch.shape
context_result = ca(batch)
context_result

tensor([[[ 0.0000,  0.0000],
         [ 0.3782, -0.4391],
         [ 0.6237, -0.6719],
         [ 0.5633, -0.6210],
         [ 0.2612, -0.2979],
         [ 0.5380, -0.6058]],

        [[ 0.3979, -0.2986],
         [ 0.5690, -0.5823],
         [ 0.3744, -0.3830],
         [ 0.3778, -0.4057],
         [ 0.5623, -0.6205],
         [ 0.2837, -0.3107]]], grad_fn=<UnsafeViewBackward0>)

In [129]:
class MultiheadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length,dropout, num_heads, bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [Casual_Attention(d_in,d_out,context_length , dropout,bias) 
                                    for _ in range(num_heads)])
        
    def forward(self, x):
        return torch.cat([head(x)  for head in self.heads] , dim=-1)
    

In [131]:
mha = MultiheadAttentionWrapper(3,1,6,0.2 , 2)
context_result = mha(batch)
context_result.shape

torch.Size([2, 6, 2])

In [199]:
class MultiheadAttention(nn.Module):
    def __init__(self, d_in, d_out,context_length, dropout, num_heads, bias=False):
        super().__init__()
        self.num_heads = num_heads
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"
        self.d_out = d_out,
        self.head_dim = d_out//num_heads
        
        self.d_in = d_in
        self.k_weights = nn.Linear(d_in,d_out , bias )
        self.q_weights = nn.Linear(d_in,d_out , bias )
        self.v_weights = nn.Linear(d_in,d_out , bias )
        self.proj = nn.Linear(d_out,d_out)
        
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('multi_mask' , torch.tril(torch.ones(context_length,context_length), diagonal=0))
        
    def forward(self,x):
        b, num_tokens, d_in = x.shape
        keys= self.k_weights(x)
        queries= self.q_weights(x)
        values= self.v_weights(x)
        
        
        
        keys = keys.view(b,num_tokens , self.num_heads, self.head_dim)
        queries = queries.view(b,num_tokens , self.num_heads, self.head_dim)
        values = values.view(b,num_tokens , self.num_heads, self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        
        
        
        attention = queries @ keys.transpose(-1,-2)
        # return attention
        
        
        masked_attention = attention.masked_fill(self.multi_mask ==0 ,-torch.inf)
        atten_soft = torch.softmax(masked_attention/keys.shape[-1]**0.5 , dim=-1)
        print(f"attention soft shape {atten_soft.shape , values.shape}")
        
        context_vec = (atten_soft @ values).transpose(1,2)
        
        context_vec = context_vec.contiguous().view(b,num_tokens,-1)
        
        return context_vec
        
        
        
    

In [201]:
mha = MultiheadAttention(3,3,6,0.0,3)
context_result = mha(batch)
context_result.shape



attention soft shape (torch.Size([2, 3, 6, 6]), torch.Size([2, 3, 6, 1]))


torch.Size([2, 6, 3])